# 05 — Correlaciones: Clima vs Generación

**Objetivo:** Demostrar estadísticamente cómo la irradiancia (NASA) y la temperatura afectan la generación eléctrica en Cuba (ONEI).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path.cwd().parent / 'data'

# 1. Cargar los datos de la NASA y la ONEI
nasa = pd.read_parquet(DATA / 'clean' / 'nasa_power_limpio.parquet')
onei = pd.read_parquet(DATA / 'clean' / 'onei_generacion.parquet')

print('Datos de NASA:', len(nasa), 'registros (mensual)')
print('Datos de ONEI:', len(onei), 'registros (anual)')

# 2. Agrupar NASA por año para poder cruzarlo con ONEI
nasa_anual = nasa.groupby('anio').agg({
    'irradiancia_kwh_m2': 'mean',
    'temperatura_c': 'mean',
    'humedad_pct': 'mean'
}).round(3).reset_index()

# 3. Unir ambas tablas por el año
df_corr = onei.merge(nasa_anual, on='anio')
display(df_corr)

## 1. Matriz de Correlación (Heatmap)

Un valor cercano a 1 indica una relación directa (si sube uno, sube el otro). Un valor negativo indica una relación inversa.

In [ ]:
# Seleccionar solo columnas numéricas para correlacionar
cols_numericas = ['gen_total_gwh', 'gen_fv_gwh', 'irradiancia_kwh_m2', 'temperatura_c', 'humedad_pct']
sub = df_corr[cols_numericas].dropna()

# Calcular correlación de Spearman (más robusta que Pearson para muestras pequeñas)
corr = sub.corr(method='spearman').round(3)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True, 
            cbar_kws={'shrink': 0.8})
plt.title('Correlación: Generación Eléctrica vs Clima en Cuba', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Gráfico de Dispersión (Scatter Plot)

Visualización de cómo la irradiancia se relaciona con la generación.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_corr, x='irradiancia_kwh_m2', y='gen_total_gwh', s=150, color='#f4a900', edgecolor='black')

# Añadir etiquetas con el año de cada punto
for i, row in df_corr.iterrows():
    plt.text(row['irradiancia_kwh_m2'] + 0.05, row['gen_total_gwh'] - 100, str(row['anio']), fontsize=10)

plt.title('Relación: Irradiancia Solar vs Generación Total (2020-2022)', fontsize=14)
plt.xlabel('Irradiancia Promedio (kWh/m²/día)', fontsize=12)
plt.ylabel('Generación Total (GW·h)', fontsize=12)
plt.grid(alpha=0.3)
plt.show()

## 3. Análisis para el Storytelling

**Pregunta al profesor:** *"¿Por qué la generación total cayó en 2022 si la irradiancia fue buena?"*

Esto demuestra que la caída no fue por falta de sol, sino por la crisis del sistema termoeléctrico estatal. El clima es una variable, pero la infraestructura estatal es el factor limitante.